# Exploring the Kigali Sentinel-2 images

**Machine Learning for Earth Observation · Year 3 · getting started (≈30 minutes)**

Before doing any machine learning, we look at the data itself. In this notebook you will:

1. list the image files and understand their names,
2. open one image file and read its information (size, pixel size, coordinates),
3. look at the pixel values and show the image,
4. combine three bands into a colour picture,
5. compute NDVI from two bands,
6. check that all files share the same pixel grid,
7. read the scene classification layer (clouds, water, vegetation),
8. compare the same place on three dates,
9. look at the values of single pixels.

Every step uses only a few lines of plain Python. Run the cells from top to bottom (Shift + Enter) and read the comments.

## 1. The tools

* **rasterio** opens satellite images stored as GeoTIFF files.
* **NumPy** does calculations on the pixel values (an image is a NumPy *array*).
* **matplotlib** draws pictures and graphs.
* **pandas** handles tables.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

DATA = "../data"          # the folder with the images, one level up from this notebook

## 2. What files do we have?

`os.listdir` gives the names of all files in a folder. We keep the `.tiff` files and sort them.

In [ ]:
files = sorted(f for f in os.listdir(DATA) if f.endswith(".tiff"))
print("number of image files:", len(files))
for f in files[:5]:           # show the first five names
    print(f)

A file name has three parts:

```
2026-09-03-00_00_2026-09-03-23_59_Sentinel-2_L2A_B04_(Raw).tiff
└── date ─┘                        └─ product ──┘ └ band ┘
```

* the **date** of the image (here 3 September 2026),
* the **product**: Sentinel-2, processing level L2A (surface reflectance),
* the **layer**: a band (B01 … B12), or a picture (NDVI, NDWI, True_color, Scene_classification_map).

How many files are there for each date? We count them with a simple loop.

In [ ]:
for date in ["2026-01-21", "2026-05-13", "2026-09-03"]:
    n = 0
    for f in files:
        if f.startswith(date):
            n = n + 1
    print(date, ":", n, "files")

## 3. Open one band and read its information

We build the full file name of band **B04** (red) on 3 September 2026 and open it.
`with rasterio.open(...) as src:` opens the file and closes it again at the end of the block.

In [ ]:
date = "2026-09-03"
band = "B04"
path = f"{DATA}/{date}-00_00_{date}-23_59_Sentinel-2_L2A_{band}_(Raw).tiff"
print(path)

with rasterio.open(path) as src:
    print("columns x rows     :", src.width, "x", src.height)
    print("number of layers   :", src.count)
    print("data type          :", src.dtypes[0])
    print("coordinate system  :", src.crs)
    print("pixel size (m)     :", src.res)
    print("bounds (m)         :", src.bounds)
    red = src.read(1)          # layer 1 as a NumPy array

What does this tell us?

* The image has **2380 columns and 1702 rows** of pixels.
* The **coordinate system** EPSG:32736 is UTM zone 36 South: positions are in **metres**.
* Each pixel is about **20 m × 20 m**.
* The **bounds** are the edges of the image in metres. From them we can compute the size of the area:

In [ ]:
with rasterio.open(path) as src:
    b = src.bounds
width_km = (b.right - b.left) / 1000
height_km = (b.top - b.bottom) / 1000
print(f"the image covers {width_km:.1f} km x {height_km:.1f} km")

## 4. The pixel values

`red` is a NumPy array with one number per pixel: the **surface reflectance** in the red band (0 = nothing reflected, 1 = everything reflected).
A value of exactly **0 means "no data"** (a thin frame around the image). We replace it by `NaN` ("not a number") so that it is ignored in calculations.

In [ ]:
print("shape:", red.shape, "| data type:", red.dtype)
print("no-data pixels (value 0):", np.sum(red == 0))

red[red == 0] = np.nan       # no data -> NaN

print("smallest value:", np.nanmin(red))
print("largest value :", np.nanmax(red))
print("average value :", round(float(np.nanmean(red)), 3))

In [ ]:
plt.figure(figsize=(9, 6))
plt.imshow(red, cmap="gray", vmin=0, vmax=0.3)     # values above 0.3 are shown white
plt.colorbar(label="reflectance (red band)")
plt.title("Band B04 (red), 3 September 2026")
plt.show()

Most pixels have a red reflectance between 0.05 and 0.25. Dark areas are water and dense vegetation (they absorb red light); bright areas are bare soil and the city. A **histogram** shows how often each value occurs:

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist(red[np.isfinite(red)], bins=100, range=(0, 0.4), color="firebrick")
plt.xlabel("reflectance (red band)"); plt.ylabel("number of pixels")
plt.show()

## 5. A colour picture from three bands

A normal photo has three colours: red, green and blue. Sentinel-2 records them in bands **B04 (red), B03 (green) and B02 (blue)**. We open the three files with a loop and keep the arrays in a dictionary.

In [ ]:
bands = {}                                  # an empty dictionary: band name -> array
for band in ["B02", "B03", "B04"]:
    path = f"{DATA}/{date}-00_00_{date}-23_59_Sentinel-2_L2A_{band}_(Raw).tiff"
    with rasterio.open(path) as src:
        a = src.read(1)
    a[a == 0] = np.nan
    bands[band] = a
    print(band, "loaded, average reflectance", round(float(np.nanmean(a)), 3))

To display the picture we put the three arrays on top of each other (`np.dstack`) in the order red, green, blue. Reflectance values are small, so we brighten them: dividing by 0.25 makes a reflectance of 0.25 fully bright, and `np.clip` keeps everything between 0 and 1.

In [ ]:
rgb = np.dstack([bands["B04"], bands["B03"], bands["B02"]])     # rows x columns x 3
rgb = np.clip(rgb / 0.25, 0, 1)

plt.figure(figsize=(10, 7))
plt.imshow(rgb)
plt.title("True colour, 3 September 2026")
plt.show()

## 6. NDVI: a vegetation index from two bands

Healthy leaves reflect a lot of near-infrared light (band **B08**) and absorb red light (B04). The **NDVI** combines the two:

NDVI = (NIR − red) / (NIR + red)

It is close to 0.6–0.8 for dense vegetation, around 0.1–0.2 for soil and buildings, and below 0 for water.

In [ ]:
path = f"{DATA}/{date}-00_00_{date}-23_59_Sentinel-2_L2A_B08_(Raw).tiff"
with rasterio.open(path) as src:
    nir = src.read(1)
nir[nir == 0] = np.nan

ndvi = (nir - bands["B04"]) / (nir + bands["B04"])

plt.figure(figsize=(10, 7))
plt.imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
plt.colorbar(label="NDVI")
plt.title("NDVI, 3 September 2026")
plt.show()
print("median NDVI of the scene:", round(float(np.nanmedian(ndvi)), 2))

The file `..._NDVI.tiff` in the folder is **not** NDVI values: it is a coloured *picture* of NDVI made by the Copernicus Browser (3 layers of colours). You can check it yourself:

In [ ]:
path = f"{DATA}/{date}-00_00_{date}-23_59_Sentinel-2_L2A_NDVI.tiff"
with rasterio.open(path) as src:
    print("layers:", src.count, "| values from", src.read().min(), "to", src.read().max())

Three layers of colour values between 0 and 1: useful to look at, but for calculations we always compute indices from the bands, as above.

## 7. Do all files share the same pixel grid?

Machine learning compares pixels **across bands and dates**. That only works if pixel (row, column) is the **same place on the ground** in every file. We check the size, pixel size and upper-left corner of a few files.

In [ ]:
for date_ in ["2026-01-21", "2026-05-13", "2026-09-03"]:
    for layer in ["B02_(Raw)", "B01_(Raw)", "Scene_classification_map_"]:
        path = f"{DATA}/{date_}-00_00_{date_}-23_59_Sentinel-2_L2A_{layer}.tiff"
        with rasterio.open(path) as src:
            print(date_, layer.ljust(26), src.width, "x", src.height,
                  "| pixel", round(src.res[0], 2), "m | corner", round(src.bounds.left), round(src.bounds.top))

All files have the same size, pixel size and corner: they are on **one grid**. This was a choice made when the images were exported: every band was resampled to 20 m. Sentinel-2 records B02, B03, B04 and B08 at 10 m and B01 and B09 at only 60 m, so those bands were made coarser or finer. The *stored* pixel size is not always the *real* resolution: compare a 10 m band with a 60 m band on a small area.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
for a, band, native in zip(ax, ["B02", "B01"], [10, 60]):
    path = f"{DATA}/2026-09-03-00_00_2026-09-03-23_59_Sentinel-2_L2A_{band}_(Raw).tiff"
    with rasterio.open(path) as src:
        img = src.read(1)
    a.imshow(img[700:775, 600:675], cmap="gray")        # a 1.5 km x 1.5 km window
    a.set_title(f"{band}: recorded at {native} m, stored at 20 m")
    a.set_xticks([]); a.set_yticks([])
plt.show()

## 8. The scene classification layer (SCL)

With every Sentinel-2 L2A image comes a **scene classification**: for each pixel, a code for what the processing found there, for example vegetation, water, cloud or cloud shadow. We use it to remove clouds before training models.

The Copernicus Browser exports the SCL as a **coloured picture** with 3 layers (red, green, blue). First we simply look at it:

In [ ]:
path = f"{DATA}/2026-01-21-00_00_2026-01-21-23_59_Sentinel-2_L2A_Scene_classification_map_.tiff"
with rasterio.open(path) as src:
    print("layers:", src.count)
    scl_picture = src.read()                     # shape: (3 layers, rows, columns)
print(scl_picture.shape)

plt.figure(figsize=(10, 7))
plt.imshow(np.moveaxis(scl_picture, 0, -1))      # imshow wants (rows, columns, 3)
plt.title("Scene classification of 21 January 2026, as exported")
plt.show()

Each colour stands for one class. The table below is the official colour legend. To turn the picture into class **codes**, we look for the pixels that have each colour.
The picture stores colours as numbers from 0 to 1, so we first multiply by 255 to get the usual 0–255 colour values.

In [ ]:
SCL_CLASSES = {                      # code: (name, (red, green, blue))
    0: ("no data",              (0, 0, 0)),
    1: ("saturated / defective", (255, 0, 0)),
    2: ("dark area",             (47, 47, 47)),
    3: ("cloud shadow",          (100, 50, 0)),
    4: ("vegetation",            (0, 160, 0)),
    5: ("not vegetated",         (255, 230, 90)),
    6: ("water",                 (0, 0, 255)),
    7: ("unclassified",          (128, 128, 128)),
    8: ("cloud, medium prob.",   (192, 192, 192)),
    9: ("cloud, high prob.",     (255, 255, 255)),
    10: ("thin cirrus",          (100, 200, 255)),
    11: ("snow / ice",           (255, 150, 255)),
}

colours = np.round(scl_picture * 255)                 # 0-1 -> 0-255
r, g, b = colours[0], colours[1], colours[2]
scl = np.zeros(r.shape, dtype=int)                    # start with code 0 (no data) everywhere
for code_, (name, (cr, cg, cb)) in SCL_CLASSES.items():
    same_colour = (r == cr) & (g == cg) & (b == cb)
    scl[same_colour] = code_

for code_, (name, colour) in SCL_CLASSES.items():
    share = 100 * np.mean(scl == code_)
    if share > 0:
        print(f"{code_:2d} {name:22s} {share:6.2f}% of the pixels")

In January, about 4% of the image is cloud and 2% thin cirrus (the grey, white and light-blue patch in the south-west). A **validity mask** keeps only the pixels that show the land surface: vegetation (4), not vegetated (5) and water (6).

In [ ]:
valid = (scl == 4) | (scl == 5) | (scl == 6)
print(f"valid pixels in January: {100 * valid.mean():.1f}%")

plt.figure(figsize=(9, 6))
plt.imshow(valid, cmap="gray")
plt.title("valid pixels on 21 January 2026 (white = valid)")
plt.show()

## 9. The same place on three dates

Because all dates share one grid, we can compare them pixel by pixel. We compute NDVI for each date, remove the invalid pixels with the SCL, and show the three maps side by side.

In [ ]:
ndvi_dates = {}
for date_ in ["2026-01-21", "2026-05-13", "2026-09-03"]:
    start = f"{DATA}/{date_}-00_00_{date_}-23_59_Sentinel-2_L2A_"

    with rasterio.open(start + "B04_(Raw).tiff") as src:
        red_ = src.read(1)
    with rasterio.open(start + "B08_(Raw).tiff") as src:
        nir_ = src.read(1)
    with rasterio.open(start + "Scene_classification_map_.tiff") as src:
        c = np.round(src.read() * 255)
    red_[red_ == 0] = np.nan
    nir_[nir_ == 0] = np.nan

    ok = ((c[0] == 0) & (c[1] == 160) & (c[2] == 0)) | ((c[0] == 255) & (c[1] == 230) & (c[2] == 90)) \
         | ((c[0] == 0) & (c[1] == 0) & (c[2] == 255))            # SCL vegetation, not vegetated or water
    nd = (nir_ - red_) / (nir_ + red_)
    nd[~ok] = np.nan                                             # hide clouds, shadows and no data
    ndvi_dates[date_] = nd
    print(date_, "median NDVI:", round(float(np.nanmedian(nd)), 2))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for a, (date_, nd) in zip(ax, ndvi_dates.items()):
    a.imshow(nd, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
    a.set_title("NDVI " + date_); a.set_xticks([]); a.set_yticks([])
plt.show()

January and May are in the rainy seasons: most of the landscape is green. On 3 September, at the end of the dry season, many fields and grasslands are dry or bare. The same pixel can therefore belong to different classes at different dates. Keep this in mind when you label training data.

## 10. Looking at single pixels

The file `kigali_training_polygons.csv` lists 1,225 labelled pixels: their class, and their **row** and **column** in the images. Reading the value of a band at a pixel is just indexing the array: `array[row, column]`.

In [ ]:
samples = pd.read_csv(f"{DATA}/kigali_training_polygons.csv")
print(samples[["obs_id", "class", "row", "col"]].head())
print()
print(samples["class"].value_counts())

We take the first pixel of each class and read its reflectance in all 12 bands of 3 September. The result is a **spectral signature**: how much light the surface reflects at each wavelength.

In [ ]:
ALL_BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"]
WAVELENGTH = [443, 490, 560, 665, 705, 740, 783, 842, 865, 945, 1610, 2190]    # nm

september = {}
for band in ALL_BANDS:
    path = f"{DATA}/2026-09-03-00_00_2026-09-03-23_59_Sentinel-2_L2A_{band}_(Raw).tiff"
    with rasterio.open(path) as src:
        september[band] = src.read(1)

plt.figure(figsize=(9, 4.5))
for cls in samples["class"].unique():
    first = samples[samples["class"] == cls].iloc[0]          # first pixel of this class
    values = [september[band][first["row"], first["col"]] for band in ALL_BANDS]
    plt.plot(WAVELENGTH, values, "o-", label=f"{cls} (row {first['row']}, col {first['col']})")
plt.xlabel("wavelength (nm)"); plt.ylabel("reflectance"); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.show()

Single pixels are noisy: this water pixel, for example, comes from a muddy part of a lake and is brighter than clear water. In the practicals we therefore average many pixels of each class.

Where is a pixel on the ground? The image *transform* links row and column to coordinates in metres: the centre of pixel (row, col) lies at

x = left edge + (col + 0.5) × pixel width  y = top edge − (row + 0.5) × pixel height

In [ ]:
first = samples.iloc[0]
with rasterio.open(path) as src:
    left, top = src.bounds.left, src.bounds.top
    pixel_w, pixel_h = src.res
x = left + (first["col"] + 0.5) * pixel_w
y = top - (first["row"] + 0.5) * pixel_h
print(f"pixel row {first['row']}, col {first['col']} ({first['class']}) is at x = {x:.0f} m, y = {y:.0f} m (UTM 36S)")
print("the table says:", first["x"], first["y"])

## 11. What you have learned, and a small helper for later

You opened the files with three lines each time:

```python
with rasterio.open(path) as src:
    a = src.read(1)
a[a == 0] = np.nan
```

In the later practicals we often need many bands, so we put these lines into a small **function** of our own. It does exactly what you did above:

In [ ]:
def load_band(date, band):
    """Open one Sentinel-2 band of one date; return reflectance with NaN for no data."""
    path = f"{DATA}/{date}-00_00_{date}-23_59_Sentinel-2_L2A_{band}_(Raw).tiff"
    with rasterio.open(path) as src:
        a = src.read(1)
    a[a == 0] = np.nan
    return a

b08_may = load_band("2026-05-13", "B08")
print("B08 on 13 May:", b08_may.shape, "| average", round(float(np.nanmean(b08_may)), 3))